# Agentic RAG | Domain Applications

In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [4]:
# Simulated document store
docs = [
    Document(page_content="LangGraph uses a StateGraph to define agent workflows. Nodes are "
             "functions, edges define transitions. Use START/END for entry/exit points.",
             metadata={"source": "concepts"}),
    Document(page_content="The Command object allows nodes to route dynamically. Import from "
             "langgraph.types. Use Command(goto='node_name', update={...}).",
             metadata={"source": "concepts"}),
    Document(page_content="Error ERR-4012: State schema mismatch. Ensure all TypedDict fields "
             "have matching types. Check NotRequired annotations.",
             metadata={"source": "errors"}),
    Document(page_content="Config: set recursion_limit in graph.compile(). Default is 25. "
             "Set LANGGRAPH_TRACING=true for debug logging.",
             metadata={"source": "config"}),
]
vector_store = InMemoryVectorStore.from_documents(docs, embeddings)

@tool
def vector_search(query: str) -> str:
    """Semantic search over documentation. Best for conceptual questions like
    'how does X work?' or 'what is the purpose of Y?'"""
    results = vector_store.similarity_search(query, k=3)
    if not results:
        return "No results found."
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in results)

@tool
def keyword_search(query: str) -> str:
    """Exact keyword search. Best for error codes (ERR-XXXX), config keys,
    specific class/function names, or version numbers."""
    query_lower = query.lower()
    matches = [d for d in docs if query_lower in d.page_content.lower()]
    if not matches:
        return "No exact matches found."
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in matches)

@tool
def web_search(query: str) -> str:
    """Search the web for latest release notes, changelogs, or community discussions.
    Use when docs are outdated or the question is about recent changes."""
    # In production: use DuckDuckGo, Tavily, or Google Search API
    return (f"[Web result for '{query}']: LangGraph v0.4.0 released 2025-03-15. "
            f"New features: streaming support for Command, improved checkpointing. "
            f"Breaking change: create_react_agent moved to langchain.agents.create_agent.")

In [5]:
agent = create_agent(
    model=model,
    tools=[vector_search, keyword_search, web_search],
    system_prompt=(
        "You are a technical documentation assistant for LangGraph/LangChain.\n\n"
        "ROUTING STRATEGY — choose the right tool based on query type:\n"
        "- Conceptual questions ('how does X work?') -> vector_search\n"
        "- Error codes, config keys, exact names -> keyword_search\n"
        "- Latest releases, changelogs, recent changes -> web_search\n"
        "- Complex questions may need MULTIPLE sources — search, evaluate, search again.\n\n"
        "RETRY STRATEGY — if retrieved results are not relevant (low confidence):\n"
        "1. Reformulate the query with more specific terms\n"
        "2. Try a DIFFERENT search tool (e.g., if vector_search returns poor results, "
        "try keyword_search with exact terms)\n"
        "3. Combine results from multiple tools before giving up\n"
        "Max 3 retrieval attempts per question before answering with what you have."
    )
)

In [6]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "I'm getting ERR-4012. What does it mean and was it fixed in the latest release?"}]
})
print(result["messages"][-1].content)

The error code ERR-4012 indicates a "State schema mismatch." This means that there is a discrepancy in the types of the fields of a `TypedDict`. You need to verify that all fields have matching types and pay attention to `NotRequired` annotations.

As for whether it was fixed in the latest release, the search results for the latest release of LangGraph, version 0.4.0, did not specifically mention a fix for ERR-4012. The release focused on new features like streaming support for Command and some structural changes in the codebase. If this error is still affecting your work, it may not have been addressed in the latest release based on the available information.
